In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from tfmap import Atlus
import numpy as np
import polars as pl

In [ ]:
def atlus_to_df(map_obj: Atlus, cell_type: str) -> pl.DataFrame:
    wn = np.linspace(650, 4000, 3475)
    pixel_idx, pixel_pos = list(zip(*map_obj.pixels.items()))
    pixel_x, pixel_y = list(zip(*pixel_pos))
    pixel_df = pl.DataFrame(dict(idx=pixel_idx, pixel_x=pixel_x, pixel_y=pixel_y))

    spectra_idx, spectra = list(zip(*map_obj.spectra_dict.items()))
    spectra_df = pl.DataFrame(np.array(spectra))
    spectra_df.columns = [f"wavenumber_{x:.2f}" for x in wn]
    spectra_df = spectra_df.with_columns(pl.Series(name="idx", values=spectra_idx))

    return (
        pixel_df.join(spectra_df, on="idx")
        .drop("idx")
        .with_columns(cell_type=pl.lit(cell_type))
    )